In [2]:
"""Find stronger revolving-door leads than Chamberlin."""
import duckdb

con = duckdb.connect("../output/investigation.duckdb", read_only=True)

# ── Q1: Former members of Congress ────────────────────────────────────────────
print("=" * 80)
print("Q1: Lobbyists who appear to be FORMER MEMBERS OF CONGRESS")
print("=" * 80)

q1 = """
SELECT
    sl.lobbyist_name,
    sl.covered_position,
    sf.registrant_name AS firm,
    COUNT(DISTINCT sf.client_name) AS num_clients,
    COUNT(DISTINCT sf.filing_uuid) AS num_filings,
    ROUND(SUM(sf.income)/1e6, 2) AS firm_income_M,
    MIN(sf.filing_year) AS first_filing_year,
    MAX(sf.filing_year) AS last_filing_year
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
WHERE (
    LOWER(sl.covered_position) LIKE '%congressman%'
    OR LOWER(sl.covered_position) LIKE '%congresswoman%'
    OR LOWER(sl.covered_position) LIKE 'representative %'
    OR LOWER(sl.covered_position) LIKE '%u.s. senator%'
    OR LOWER(sl.covered_position) LIKE '%us senator%'
    OR LOWER(sl.covered_position) LIKE 'senator %'
    OR LOWER(sl.covered_position) LIKE '%member of congress%'
    OR LOWER(sl.covered_position) LIKE '%house of representatives%'
)
AND LOWER(sl.covered_position) NOT LIKE '%aide%'
AND LOWER(sl.covered_position) NOT LIKE '%chief of staff%'
AND LOWER(sl.covered_position) NOT LIKE '%counsel%'
AND LOWER(sl.covered_position) NOT LIKE '%advisor%'
AND LOWER(sl.covered_position) NOT LIKE '%legislative%'
AND LOWER(sl.covered_position) NOT LIKE '%staff director%'
AND LOWER(sl.covered_position) NOT LIKE '%press%'
AND sl.lobbyist_name IS NOT NULL
GROUP BY sl.lobbyist_name, sl.covered_position, sf.registrant_name
HAVING COUNT(DISTINCT sf.client_name) >= 3
ORDER BY firm_income_M DESC NULLS LAST
LIMIT 30
"""
for row in con.execute(q1).fetchall():
    print(row)

# ── Q2: Fast pivots (recent government roles) ─────────────────────────────────
print()
print("=" * 80)
print("Q2: FAST PIVOTS — covered_position mentions 2022/2023/2024")
print("=" * 80)

q2 = """
SELECT
    sl.lobbyist_name,
    sl.covered_position,
    sf.registrant_name AS firm,
    MIN(sf.filing_year) AS first_lobbying_year,
    COUNT(DISTINCT sf.client_name) AS num_clients,
    ROUND(SUM(sf.income)/1e6, 2) AS firm_income_M
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
WHERE (
    sl.covered_position LIKE '%2024%'
    OR sl.covered_position LIKE '%2023%'
    OR sl.covered_position LIKE '%2022%'
)
AND sl.lobbyist_name IS NOT NULL
GROUP BY sl.lobbyist_name, sl.covered_position, sf.registrant_name
HAVING COUNT(DISTINCT sf.client_name) >= 2
ORDER BY first_lobbying_year DESC, num_clients DESC
LIMIT 30
"""
for row in con.execute(q2).fetchall():
    print(row)

# ── Q3: Chamberlin's client portfolio ─────────────────────────────────────────
print()
print("=" * 80)
print("Q3: Chamberlin's client portfolio (for the record)")
print("=" * 80)

q3 = """
SELECT
    sf.client_name,
    COUNT(DISTINCT sf.filing_uuid) AS filings,
    ROUND(SUM(sf.income), 0) AS total_disclosed_income,
    STRING_AGG(DISTINCT sa.general_issue_code, ', ') AS issue_codes,
    MIN(sf.filing_year) AS first_year,
    MAX(sf.filing_year) AS last_year
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
LEFT JOIN senate_activities sa USING (filing_uuid)
WHERE sl.lobbyist_name ILIKE '%chamberlin%'
  AND sl.covered_position ILIKE '%commerce%'
GROUP BY sf.client_name
ORDER BY total_disclosed_income DESC NULLS LAST
LIMIT 20
"""
for row in con.execute(q3).fetchall():
    print(row)

con.close()

Q1: Lobbyists who appear to be FORMER MEMBERS OF CONGRESS
('JOHN KINGSTON', '1993-2015, Congressman-U.S. House of Representatives', 'SQUIRE PATTON BOGGS', 31, 312, 32.33, 2022, 2026)
('GREG WALDEN', 'US Congressman 1999-2021', 'ALPINE GROUP PARTNERS, LLC.', 21, 114, 27.1, 2022, 2026)
('ALAN WHEAT', 'Former Member of Congress', 'WHEAT SHROYER GOVERNMENT RELATIONS LLC', 13, 138, 24.28, 2022, 2026)
('MARTHA ROBY', 'Former Member of U.S. House of Representatives', 'BRADLEY ARANT BOULT CUMMINGS LLP', 29, 241, 21.3, 2022, 2026)
('REBEKAH SUNGALA', 'Jan. 2018 - Jan. 2019: Leg. Assist., House Transportation & Infrastructure Committee; Sept. 2017-Jan. 2019: Scheduler/Executive Assistant; Congressman Bill Shuster; Nov. 2012-Aug. 2017: Sr. Field Rep., Congressman Bill Shuster', 'SQUIRE PATTON BOGGS', 24, 161, 15.17, 2022, 2026)
('STEPHANIE CHEUNG', 'February 2021-May 2021: Intern, Office of Congresswoman Grace Meng (D-NY-06)', 'SQUIRE PATTON BOGGS', 16, 79, 15.04, 2022, 2026)
('GREG WALDEN', 'US 

In [3]:
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

q4 = """
SELECT
    sf.client_name,
    COUNT(DISTINCT sf.filing_uuid) AS filings,
    ROUND(SUM(sf.income), 0) AS total_disclosed_income,
    STRING_AGG(DISTINCT sa.general_issue_code, ', ') AS issue_codes,
    MIN(sf.filing_year || '-' || sf.filing_period) AS first_filing,
    MAX(sf.filing_year || '-' || sf.filing_period) AS last_filing
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
LEFT JOIN senate_activities sa USING (filing_uuid)
WHERE sl.lobbyist_name ILIKE '%bridenstine%'
GROUP BY sf.client_name
ORDER BY total_disclosed_income DESC NULLS LAST;
"""
for row in con.execute(q4).fetchall():
    print(row)

con.close()

('UNITED LAUNCH ALLIANCE', 6, 19520000.0, 'SCI, BUD, DEF, AER', '2025-first_quarter', '2026-first_quarter')
('ALL POINTS LLC', 7, 3830000.0, 'AVI, DEF, AER, SCI, BUD', '2024-fourth_quarter', '2026-first_quarter')
('UNIVERSITY OF ARIZONA', 6, 2050000.0, 'DEF, AER, SCI', '2025-first_quarter', '2026-first_quarter')
('AGILE SPACE INDUSTRIES', 7, 1740000.0, 'AVI, SCI, BUD, DEF, AER', '2024-fourth_quarter', '2026-first_quarter')
('VOYAGER SPACE HOLDINGS', 7, 1160000.0, 'AVI, BUD, SCI, AER', '2024-fourth_quarter', '2026-first_quarter')
('VIASAT, INC.', 5, 840000.0, 'AER, TEC, COM, DEF', '2025-fourth_quarter', '2026-first_quarter')
('CHOCTAW NATION OF OKLAHOMA', 4, 560000.0, 'SCI, AVI, AER, BUD', '2024-fourth_quarter', '2025-second_quarter')
('REDWIRE', 4, 320000.0, 'AER, AVI, BUD, SCI', '2024-fourth_quarter', '2025-third_quarter')
('IMPULSE SPACE', 3, 320000.0, 'AVI, AER, SCI, BUD', '2024-fourth_quarter', '2025-third_quarter')
('BLACK MOON ENERGY CORPORATION', 4, 240000.0, 'AER, ENG, SCI', '2

In [9]:
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

q5 = """
SELECT DISTINCT
    sf.client_name,
    sa.general_issue_code,
    sa.description
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
JOIN senate_activities sa USING (filing_uuid)
WHERE sl.lobbyist_name ILIKE '%bridenstine%'
  AND sa.description IS NOT NULL
ORDER BY sf.client_name, sa.general_issue_code
LIMIT 100;
"""

for row in con.execute(q5).fetchall():
    print(row)

con.close()

('AGILE SPACE INDUSTRIES', 'AER', 'Aerospace and aviation issues generally')
('AGILE SPACE INDUSTRIES', 'AER', 'Monitor space issues generally')
('AGILE SPACE INDUSTRIES', 'AER', 'Monitor space issues generally; increase funding for multi-mode propulsion')
('AGILE SPACE INDUSTRIES', 'AVI', 'Aerospace and aviation issues generally')
('AGILE SPACE INDUSTRIES', 'BUD', 'Monitor space issues generally, increase funding for multi-mode propulsion')
('AGILE SPACE INDUSTRIES', 'BUD', 'Aerospace and aviation issues generally')
('AGILE SPACE INDUSTRIES', 'DEF', 'Monitor space issues generally; increase funding for multi-mode propulsion')
('AGILE SPACE INDUSTRIES', 'SCI', 'Aerospace and aviation issues generally')
('ALL POINTS LLC', 'AER', 'Aerospace and aviation issues generally')
('ALL POINTS LLC', 'AER', 'Monitor space issues generally')
('ALL POINTS LLC', 'AER', 'Monitor space issues generally; secure funding for payload processing facilities; NDAA; EUL opportunities')
('ALL POINTS LLC', 'AER'

In [8]:
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

q6 = """
SELECT
    sge.entity_name,
    COUNT(DISTINCT sf.filing_uuid) AS filings,
    STRING_AGG(DISTINCT sf.client_name, ', ') AS clients
FROM senate_lobbyists sl
JOIN senate_filings sf USING (filing_uuid)
JOIN senate_gov_entities sge USING (filing_uuid)
WHERE sl.lobbyist_name ILIKE '%bridenstine%'
GROUP BY sge.entity_name
ORDER BY filings DESC;
"""
for row in con.execute(q6).fetchall():
    print(row)

con.close()

('HOUSE OF REPRESENTATIVES', 40, 'VOYAGER SPACE HOLDINGS, UNIVERSITY OF ARIZONA, IMPULSE SPACE, SPECIAL AEROSPACE SERVICES (SAS), REDWIRE, BLACK MOON ENERGY CORPORATION, ALL POINTS LLC, UNITED LAUNCH ALLIANCE, VIASAT, INC., CHOCTAW NATION OF OKLAHOMA, AGILE SPACE INDUSTRIES')
('SENATE', 40, 'AGILE SPACE INDUSTRIES, UNITED LAUNCH ALLIANCE, VIASAT, INC., ALL POINTS LLC, VOYAGER SPACE HOLDINGS, UNIVERSITY OF ARIZONA, IMPULSE SPACE, SPECIAL AEROSPACE SERVICES (SAS), REDWIRE, BLACK MOON ENERGY CORPORATION')
('Natl Aeronautics & Space Administration (NASA)', 24, 'REDWIRE, EXLABS, BLACK MOON ENERGY CORPORATION, VOYAGER SPACE HOLDINGS, UNIVERSITY OF ARIZONA, UNIVERSITY OF OKLAHOMA, IMPULSE SPACE, UNITED LAUNCH ALLIANCE, ALL POINTS LLC')
('Defense, Dept of (DOD)', 16, 'SPECIAL AEROSPACE SERVICES (SAS), UNIVERSITY OF ARIZONA, UNITED LAUNCH ALLIANCE, ALL POINTS LLC, AGILE SPACE INDUSTRIES')
('Executive Office of the President (EOP)', 16, 'REDWIRE, BLACK MOON ENERGY CORPORATION, VOYAGER SPACE HOLD